In [ ]:
from __future__ import annotations
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(
    model = "gpt-5.6-luna",
    base_url='https://api.gapgpt.app/v1',
    api_key=os.getenv("GAPGPT_API_KEY")
)

messages = "Hi there, My name is Dr.Alireza"

llm.invoke(messages) 

In [ ]:
from openai import OpenAI


# ایجاد یک نمونه از کلاینت با کلید API خود
client = OpenAI(base_url='https://api.gapgpt.app/v1', api_key=os.getenv("GAPGPT_API_KEY"))

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "user", "content": "salam dadash, chetori?!"}
    ]
)

print(response.choices[0].message.content)


In [6]:
from dotenv import find_dotenv
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from models import luna_model
from langgraph.checkpoint.memory import InMemorySaver
import os
from langchain_openai import ChatOpenAI

load_dotenv(find_dotenv())
sh_memory=InMemorySaver()

api_key_test = os.getenv("GOOGLE_API_KEY")
# api = api_key_test.removesuffix("test")
model = ChatGoogleGenerativeAI(model="gemini-3.7-flash", api_key=api_key_test)
# messages = [HumanMessage("hey there, what's up")]

print(api_key_test[-8:])


hkIQtest


In [7]:
from langchain.agents.middleware import wrap_model_call, ModelResponse, ModelRequest
from typing import Callable

@wrap_model_call
def retry_on_failure(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    try:
        return handler(request)
    
    except Exception:
        fallback_request = request.override(model=luna_model)
        return handler(fallback_request)

In [8]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=model,
    system_prompt="this is a test. want to see if this message would be sent or not..",
    middleware=[retry_on_failure]
)

In [9]:
thread = {"configurable":{"thread_id": "1"}}
result = agent.invoke({"messages": [HumanMessage("hey, how you doin?")]}, config=thread)
last = result["messages"][-1]
print(last.response_metadata.get("model_name"))

gpt-5.6-luna


In [11]:
print(last)

content='I’m doing well. Both of your messages came through.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 6164, 'total_tokens': 6180, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': None, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': None, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.6-luna', 'system_fingerprint': None, 'id': 'resp_0d6cc41b20a45ea8016a84ea45227c8195a90a9a4e5edcc59e', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a01732-95e2-7781-b717-c71155de2070-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 6164, 'output_tokens': 16, 'total_tokens': 6180, 'input_token_details': {}, 'output_token_details': {'reasoning': 0}}


In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

In [ ]:
REPO_ROOT = Path(__file__).cwd()
CHROMA_DIR = REPO_ROOT / ".chroma_index"
COLLECTION_NAME = "repo_corpus"
EMBEDDING_MODEL = "gemini-embedding-2-preview"

DOC_FILES = ["ROADMAP.md", "AGENTS.md", "README.md", "MENTOR_HANDOFF.md"]
CODE_GLOB = "src/**/*.py"

In [ ]:
def _embedding(): 
   return GoogleGenerativeAIEmbeddings(model=EMBEDDING_MODEL)

In [ ]:
def _load_source_documents() -> list[Document]:
    documents = [
        Document(
            page_content=(REPO_ROOT/name).read_text(encoding="utf-8"),
            metadata={"source": name}
        ) for name in DOC_FILES
    ]
    for path in sorted(REPO_ROOT.glob(CODE_GLOB)):
        if "__pycache__" in path.parts:
            continue
        documents.append(
            Document(
                page_content=path.read_text(encoding="utf-8"),
                metadata={"source":str(path.relative_to(REPO_ROOT))}
            )
        )
    return documents

In [ ]:
def build_index() -> Chroma:
    docs = _load_source_documents()
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
    chunks = splitter.split_documents(documents=docs)
    store = Chroma.from_documents(
        documents=chunks,
        embedding=_embedding(),
        persist_directory=str(CHROMA_DIR),
        collection_name=COLLECTION_NAME
    )
    return Chroma

In [ ]:
from langchain_core.vectorstores import VectorStoreRetriever
def get_retriever(k=3) -> VectorStoreRetriever:
    
    if not CHROMA_DIR.exists():
        raise FileNotFoundError(
            f'index not found '
        )

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from nodes.rag import *

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    (
      "sysytem",
      "answer question using web-searched context provided"  
    ),
    (
      "human",
      "question: {question}, web-searched: {context}"  
    )
    
])